# Part 3 — Deep Dive Analysis

**Dataset:** Oxford-IIIT Pet (cat vs. dog binary classification)  
**Models compared:** Scratch CNN, Deeper CNN, ResNet18, ResNet50, MobileNetV3-Small  
**Key question:** Does transfer learning outperform training from scratch on a small dataset?

---
**Dataset split:**
- Train: 3,312 images
- Validation: 368 images
- Test: 3,669 images

**Training regime for transfer learning:**  
Head-only (3 epochs, LR=1e-3) → Full finetune (22 epochs, LR=1e-4)

**Best result:** ResNet50 transfer — **99.62% test accuracy**, **99.56% macro F1**

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path(r'C:\Users\emil_\vscode\Assignment1\part_3')
OUTPUTS = NOTEBOOK_DIR / 'outputs'
COMP_DIR = OUTPUTS / 'external_model_comparison_2026-04-29_082032'

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
PALETTE = list(plt.cm.tab10.colors)

def load_json(path):
    with open(path) as f:
        return json.load(f)

def load_run(run_dir):
    run_dir = Path(run_dir)
    result = {'name': run_dir.name, 'path': run_dir}
    for fname in ('summary.json', 'config.json', 'training_history.json'):
        fpath = run_dir / fname
        if fpath.exists():
            result[fname.replace('.json', '')] = load_json(fpath)
    return result

MODEL_ORDER = ['scratch_cnn', 'deeper_cnn', 'resnet18_transfer', 'mobilenet_v3_transfer', 'resnet50_transfer']
MODEL_LABELS = ['Scratch\nCNN', 'Deeper\nCNN', 'ResNet18\nTransfer', 'MobileNetV3\nTransfer', 'ResNet50\nTransfer']
MODEL_COLORS = PALETTE[:5]

models = {name: load_run(COMP_DIR / name) for name in MODEL_ORDER}

print('Loaded models:')
for name in MODEL_ORDER:
    m = models[name]
    smry = m['summary']
    cfg  = m['config']
    print(f'  {name:<28} acc={smry["final_test_accuracy"]*100:.2f}%  params={smry["trainable_parameters"]:>10,}  time={smry["total_training_time_seconds"]/60:.1f}min')

---
## 1. Dataset Overview — Oxford-IIIT Pet

The Oxford-IIIT Pet dataset contains 7,393 images of 37 breeds (25 dog, 12 cat breeds).  
Here we use binary classification: **cat vs. dog** (species-level label from breed annotations).

**Class distribution:**
- Cats: ~1,183 test images
- Dogs: ~2,486 test images

The dataset is **imbalanced** (~32% cats, ~68% dogs), which is why macro F1 is reported alongside accuracy — accuracy alone would reward a dog-predicting baseline at 68%.

**Data augmentation applied to all models:**  
`RandomHorizontalFlip → RandomAffine(rotation=±15°, translate=10%) → ColorJitter → Normalize(ImageNet)`

All images resized to **224×224** (standard for ImageNet-pretrained models).

In [ ]:
# Dataset class balance visualization
cat_test = models['resnet50_transfer']['summary']['confusion_matrix_counts'][0]
dog_test = models['resnet50_transfer']['summary']['confusion_matrix_counts'][1]
n_cats_test = sum(cat_test)
n_dogs_test = sum(dog_test)
total_test = n_cats_test + n_dogs_test

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.bar(['Cats', 'Dogs'], [n_cats_test, n_dogs_test], color=['steelblue', 'coral'], edgecolor='black')
ax.set_title('Test Set Class Distribution')
ax.set_ylabel('Number of Images')
for x, n in enumerate([n_cats_test, n_dogs_test]):
    ax.text(x, n + 10, f'{n} ({n/total_test*100:.1f}%)', ha='center', fontsize=10)

ax = axes[1]
sizes = [n_cats_test, n_dogs_test]
ax.pie(sizes, labels=[f'Cats\n{n_cats_test}', f'Dogs\n{n_dogs_test}'],
       colors=['steelblue', 'coral'], autopct='%1.1f%%', startangle=90)
ax.set_title('Test Set Class Split')

plt.suptitle('Oxford-IIIT Pet — Cat vs. Dog Test Set (3,669 images)', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Total test images : {total_test}')
print(f'Cats              : {n_cats_test} ({n_cats_test/total_test*100:.1f}%)')
print(f'Dogs              : {n_dogs_test} ({n_dogs_test/total_test*100:.1f}%)')
print(f'Class imbalance   : {n_dogs_test/n_cats_test:.2f}:1 (dogs:cats)')

---
## 2. Model Comparison Overview

In [ ]:
# Collect all metrics
accs   = [models[n]['summary']['final_test_accuracy'] * 100 for n in MODEL_ORDER]
f1s    = [models[n]['summary']['final_test_macro_f1'] * 100 for n in MODEL_ORDER]
params = [models[n]['summary']['trainable_parameters']       for n in MODEL_ORDER]
times  = [models[n]['summary']['total_training_time_seconds'] / 60  for n in MODEL_ORDER]
epochs = [models[n]['summary']['epochs_completed']            for n in MODEL_ORDER]
img_per_sec = [models[n]['summary']['test_evaluation_images_per_second'] for n in MODEL_ORDER]

print(f'{'Model':<28} {'Test Acc':>9} {'Macro F1':>9} {'Params':>12} {'Train Time':>12} {'Epochs':>8}')
print('-' * 85)
for name, acc, f1, p, t, ep in zip(MODEL_ORDER, accs, f1s, params, times, epochs):
    print(f'{name:<28} {acc:>8.2f}% {f1:>8.2f}% {p:>12,} {t:>10.1f}min {ep:>8}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Test accuracy ---
ax = axes[0, 0]
bars = ax.bar(MODEL_LABELS, accs, color=MODEL_COLORS, edgecolor='black', linewidth=0.5)
ax.set_ylim(65, 102)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Test Accuracy by Model')
ax.axhline(80, color='red', linestyle='--', linewidth=1, alpha=0.5, label='80% reference')
ax.legend(fontsize=9)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=9)

# --- Accuracy vs params (log scale) ---
ax = axes[0, 1]
is_transfer = [models[n]['config']['transfer_learning'] for n in MODEL_ORDER]
for i, (p, a, label, tf) in enumerate(zip(params, accs, MODEL_LABELS, is_transfer)):
    marker = 's' if tf else 'o'
    ax.scatter(p, a, color=MODEL_COLORS[i], s=180, marker=marker,
               edgecolors='black', linewidths=0.7, zorder=5)
    ax.annotate(label.split('\n')[0], (p, a),
                textcoords='offset points', xytext=(5, 3), fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('Trainable Parameters (log scale)')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Accuracy vs. Model Size (■=transfer, ●=scratch)')
handles = [mpatches.Patch(color='gray', label='● Scratch'),
           mpatches.Patch(facecolor='white', edgecolor='black', label='■ Transfer')]
ax.legend(handles=handles, fontsize=9)

# --- Training time vs accuracy ---
ax = axes[1, 0]
for i, (t, a, label) in enumerate(zip(times, accs, MODEL_LABELS)):
    ax.scatter(t, a, color=MODEL_COLORS[i], s=180, edgecolors='black', linewidths=0.7, zorder=5)
    ax.annotate(label.split('\n')[0], (t, a),
                textcoords='offset points', xytext=(4, 3), fontsize=8)
ax.set_xlabel('Total Training Time (minutes)')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Accuracy vs. Training Time')

# --- Accuracy vs inference speed ---
ax = axes[1, 1]
for i, (ips, a, label) in enumerate(zip(img_per_sec, accs, MODEL_LABELS)):
    ax.scatter(ips, a, color=MODEL_COLORS[i], s=180, edgecolors='black', linewidths=0.7, zorder=5)
    ax.annotate(label.split('\n')[0], (ips, a),
                textcoords='offset points', xytext=(3, 3), fontsize=8)
ax.set_xlabel('Inference Speed (images/second)')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Accuracy vs. Inference Speed')

plt.suptitle('Section 2: Model Comparison — Oxford-IIIT Pet (Cat vs. Dog)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / 'deep_dive_model_comparison.png', bbox_inches='tight', dpi=120)
plt.show()

### Model Overview Analysis

**Dramatic accuracy gap between scratch and transfer learning:**

| Model | Test Acc | Params | Train Time |
|---|---|---|---|
| Scratch CNN | 77.08% | 422K | 21.4 min |
| Deeper CNN | 94.00% | 3.5M | 41.2 min |
| MobileNetV3 | 97.36% | 1.5M | 9.3 min |
| ResNet18 | 98.99% | 11.2M | 10.7 min |
| ResNet50 | **99.62%** | 23.5M | 13.6 min |

**Transfer learning wins decisively**, achieving near-perfect accuracy in 13 minutes, while the scratch CNN only reaches 77% after 21 minutes. The key advantage: ImageNet-pretrained weights encode 14M+ images of visual knowledge, giving the model a massive head start.

**MobileNetV3 efficiency:** 1.5M params → 97.36% accuracy in 9.3 minutes. The best accuracy-per-parameter model by far — ideal for deployment on resource-constrained devices.

---
## 3. Scratch CNN vs. Deeper CNN — Why the Gap?

The basic scratch CNN (422K params) only reaches 77%. The improved version (3.5M params, AdamW + cosine schedule + class weights) reaches 94%. This 17% gap reveals what matters most when training from scratch on small data.

In [ ]:
scratch = models['scratch_cnn']
deeper  = models['deeper_cnn']

# Training curves comparison
s_hist = scratch['training_history']
d_hist = deeper['training_history']

s_epochs = [e['epoch'] for e in s_hist]
d_epochs = [e['epoch'] for e in d_hist]

s_val_acc = [e['val_accuracy'] * 100 for e in s_hist]
d_val_acc = [e['val_accuracy'] * 100 for e in d_hist]
s_train_acc = [e['train_accuracy'] * 100 for e in s_hist]
d_train_acc = [e['train_accuracy'] * 100 for e in d_hist]
s_val_loss = [e['val_loss'] for e in s_hist]
d_val_loss = [e['val_loss'] for e in d_hist]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(s_epochs, s_train_acc, 'b--', alpha=0.5, linewidth=1.5, label='Scratch train')
ax.plot(s_epochs, s_val_acc,   'b-',              linewidth=2,   label='Scratch val')
ax.plot(d_epochs, d_train_acc, 'g--', alpha=0.5, linewidth=1.5, label='Deeper train')
ax.plot(d_epochs, d_val_acc,   'g-',              linewidth=2,   label='Deeper val')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Training Dynamics: Scratch vs. Deeper CNN')
ax.legend(fontsize=9)
ax.axhline(77.08, color='blue', linestyle=':', linewidth=1, alpha=0.5)
ax.axhline(94.00, color='green', linestyle=':', linewidth=1, alpha=0.5)

ax = axes[1]
ax.plot(s_epochs, s_val_loss, 'b-', linewidth=2, label=f'Scratch (final acc 77.1%)')
ax.plot(d_epochs, d_val_loss, 'g-', linewidth=2, label=f'Deeper (final acc 94.0%)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_title('Validation Loss: Scratch vs. Deeper CNN')
ax.legend(fontsize=9)

plt.suptitle('Section 3: Scratch CNNs — Training from Random Weights', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS / 'deep_dive_scratch_comparison.png', bbox_inches='tight', dpi=120)
plt.show()

# Confusion matrix comparison
for model_name, label in [('scratch_cnn', 'Scratch CNN (77.1%)'), ('deeper_cnn', 'Deeper CNN (94.0%)')]:
    cm = models[model_name]['summary']['confusion_matrix_counts']
    tp_cat = cm[0][0]; fp_cat = cm[0][1]
    fn_dog = cm[1][0]; tp_dog = cm[1][1]
    print(f'\n{label}:')
    print(f'  Cats correctly classified : {tp_cat} / {tp_cat+fp_cat} ({tp_cat/(tp_cat+fp_cat)*100:.1f}%)')
    print(f'  Dogs correctly classified : {tp_dog} / {fn_dog+tp_dog} ({tp_dog/(fn_dog+tp_dog)*100:.1f}%)')
    print(f'  Cats misclassified as dogs: {fp_cat} ({fp_cat/(tp_cat+fp_cat)*100:.1f}%)')
    print(f'  Dogs misclassified as cats: {fn_dog} ({fn_dog/(fn_dog+tp_dog)*100:.1f}%)')

### Scratch CNN Analysis

**Why scratch_cnn only reaches 77%:**
- **Dataset too small for the problem.** 3,312 training images across 2 classes seems sufficient, but cat/dog classification requires learning subtle discriminating features (face shape, ear type, body proportions) that are harder than MNIST digit strokes.
- **Underfitting + overfitting simultaneously.** The simple CNN learns basic texture differences but can't capture the full visual variation across 37 breeds.
- **Cats are harder.** With 77% accuracy, a disproportionate number of errors are cats misclassified as dogs — because dogs are the majority class and many cat breeds are visually ambiguous.

**Why deeper_cnn reaches 94%:**
- **More capacity** (3.5M vs 422K params) allows richer feature representations.
- **AdamW optimizer** (Adam with decoupled weight decay) provides better regularisation than plain Adam.
- **Cosine LR schedule** prevents premature convergence — LR decreases smoothly over 80 epochs.
- **Class-weighted loss** explicitly compensates for the 2:1 dog:cat imbalance, pushing the model to learn cat features better.
- **Label smoothing** (ε=0.05) prevents overconfident predictions on ambiguous cases.

Even with all these improvements, the deeper scratch CNN still can't match ResNet18's 98.99% — achieved with only 25 epochs. The **17 additional points of accuracy transfer learning provides** come almost entirely from pre-learned visual representations.

---
## 4. Transfer Learning Deep Dive

**Training protocol for all transfer models:**
1. **Head training** (3 epochs, LR=1e-3): Freeze backbone, train only new classifier head. Rapid adaptation to the new task without destroying pretrained weights.
2. **Full finetune** (22 epochs, LR=1e-4): Unfreeze all weights, train end-to-end with low LR to preserve general features while adapting to the specific dataset.

In [ ]:
transfer_models = ['resnet18_transfer', 'mobilenet_v3_transfer', 'resnet50_transfer']
transfer_labels = ['ResNet18', 'MobileNetV3', 'ResNet50']
transfer_colors = [MODEL_COLORS[2], MODEL_COLORS[3], MODEL_COLORS[4]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Validation accuracy curves (all 3 transfer models)
ax = axes[0]
for name, label, color in zip(transfer_models, transfer_labels, transfer_colors):
    hist = models[name]['training_history']
    epochs_h = [e['epoch'] for e in hist]
    val_accs  = [e['val_accuracy'] * 100 for e in hist]
    ax.plot(epochs_h, val_accs, color=color, linewidth=2, label=label, marker='o', markersize=3)

# Head vs finetune boundary
ax.axvline(3, color='black', linestyle='--', linewidth=1.5, alpha=0.6, label='Finetune starts (epoch 3)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Transfer Learning — Validation Accuracy')
ax.legend(fontsize=9)
ax.annotate('Head only', xy=(1.5, 85), ha='center', fontsize=9, color='gray')
ax.annotate('→ Finetune', xy=(10, 85), ha='center', fontsize=9, color='gray')

# Validation loss curves
ax = axes[1]
for name, label, color in zip(transfer_models, transfer_labels, transfer_colors):
    hist = models[name]['training_history']
    epochs_h = [e['epoch'] for e in hist]
    val_loss  = [e['val_loss'] for e in hist]
    ax.plot(epochs_h, val_loss, color=color, linewidth=2, label=label, marker='o', markersize=3)
ax.axvline(3, color='black', linestyle='--', linewidth=1.5, alpha=0.6)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_title('Transfer Learning — Validation Loss')
ax.legend(fontsize=9)

plt.suptitle('Section 4: Transfer Learning Training Dynamics', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS / 'deep_dive_transfer_learning.png', bbox_inches='tight', dpi=120)
plt.show()

In [ ]:
# Head vs finetune epoch analysis
print('Transfer Learning: Accuracy at Key Epochs')
print(f'{'Model':<20} {'After head (ep3)':>18} {'Best val acc':>14} {'Best epoch':>12} {'Final test':>12}')
print('-' * 80)
for name, label in zip(transfer_models, transfer_labels):
    hist = models[name]['training_history']
    smry = models[name]['summary']
    acc_ep3 = [e['val_accuracy'] for e in hist if e['epoch'] == 3]
    acc_ep3 = acc_ep3[0] * 100 if acc_ep3 else 0
    best_val = smry['best_validation_accuracy'] * 100
    best_ep  = smry['best_epoch']
    test_acc = smry['final_test_accuracy'] * 100
    print(f'{label:<20} {acc_ep3:>17.2f}% {best_val:>13.2f}% {best_ep:>12} {test_acc:>11.2f}%')

### Transfer Learning Analysis

**Head training phase (epochs 1–3):**
- All transfer models jump to ~85–95% validation accuracy *within 3 epochs*. This is remarkable: only the final classification layer is trained, yet the pretrained conv features immediately generalise to cats and dogs.
- **This proves that ImageNet features are highly transferable.** The network never saw cat/dog species labels during ImageNet training, yet its intermediate representations (edges → textures → object parts → objects) are general enough to distinguish species.

**Finetune phase (epochs 4–25):**
- Accuracy climbs further as all layers adapt. Lower LR (1e-4 vs 1e-3) prevents catastrophic forgetting — the general features are preserved while task-specific discriminations are refined.
- ResNet50 reaches **100% validation accuracy** at epoch 7 (val set = 368 images). ResNet18 reaches 99.45% at epoch 25.

**Why does a model pretrained on flowers/vehicles help with cat/dog?**
> The visual feature hierarchy is universal, not category-specific. Detecting ears, fur texture, facial structures — these emerge from learning to classify *any* visual objects. The low-level features (Gabor-like edge detectors in layer 1) are identical across all vision tasks. The mid-level features (fur textures, curved surfaces) appear in ImageNet's wide diversity. By the time we attach a new classification head, we've inherited millions of images worth of visual understanding.

---
## 5. ResNet Architecture — Residual Connections and 1×1 Convolutions

In [ ]:
# Architectural comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Params vs accuracy (all 5 models)
ax = axes[0]
for i, (name, label) in enumerate(zip(MODEL_ORDER, MODEL_LABELS)):
    p = models[name]['summary']['trainable_parameters']
    a = models[name]['summary']['final_test_accuracy'] * 100
    tf = models[name]['config']['transfer_learning']
    marker = 's' if tf else 'o'
    ax.scatter(p / 1e6, a, color=MODEL_COLORS[i], s=200, marker=marker,
               edgecolors='black', linewidths=0.7, zorder=5)
    offset = (5e4, 0.5) if name != 'mobilenet_v3_transfer' else (5e4, -1.5)
    ax.annotate(label.replace('\n', '\n'), (p / 1e6, a),
                textcoords='offset points', xytext=(5, 5 if name != 'mobilenet_v3_transfer' else -15),
                fontsize=8)
ax.set_xlabel('Parameters (millions)')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Efficiency Frontier: Accuracy vs. Parameters')
ax.set_ylim(70, 101)

# Accuracy per million parameters (efficiency metric)
ax = axes[1]
efficiency = [
    (models[n]['summary']['final_test_accuracy'] * 100) /
    (models[n]['summary']['trainable_parameters'] / 1e6)
    for n in MODEL_ORDER
]
bars = ax.bar(MODEL_LABELS, efficiency, color=MODEL_COLORS, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Accuracy (%) / Million Parameters')
ax.set_title('Parameter Efficiency (higher = more efficient)')
for bar, e in zip(bars, efficiency):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{e:.1f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Section 5: Model Architecture Efficiency', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS / 'deep_dive_efficiency.png', bbox_inches='tight', dpi=120)
plt.show()

### ResNet Architecture — Key Design Insights

**Residual (skip) connections:**
```
  input x ─────────────────────────┐
     │                             │
  [Conv → BN → ReLU]               │
  [Conv → BN        ]              │
     │                             │
     └──────── + ←── (identity) ───┘
              ReLU
              output
```
The skip connection adds the input directly to the output: `output = F(x) + x`.  
This solves **vanishing gradients** in deep networks. With a plain network, gradients become exponentially small through 50+ layers. With skip connections, the gradient can flow directly from loss back to early layers without passing through all the multiplications — the derivative of `F(x) + x` with respect to `x` is `∂F/∂x + 1`, never vanishing.

**1×1 Convolutions (bottleneck blocks in ResNet50):**
```
  256 channels
     │
  [1×1 Conv → 64 channels]    ← dimensionality reduction
     │
  [3×3 Conv → 64 channels]    ← spatial processing
     │
  [1×1 Conv → 256 channels]   ← dimensionality restoration
```
A 1×1 conv kernel receives input from **all** output channels of the previous layer simultaneously. With 64 input channels, each 1×1 kernel gets 64 inputs — one per channel at that spatial position.  
This compresses 256 channels → 64 (saves 4× computation in the 3×3 layer) then expands back — allowing ResNet50 to be much deeper than ResNet18 while remaining computationally tractable.

**ResNet50 vs. ResNet18 accuracy (99.62% vs 98.99%):**  
ResNet50 is ~2× deeper and has 3× more parameters. The bottleneck blocks allow learning more complex feature hierarchies. The 0.63% accuracy gap corresponds to ~23 fewer errors on 3,669 test images.

---
## 6. Error Analysis — Confusion Matrices

In [ ]:
import matplotlib.ticker as ticker

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
classes = ['Cat', 'Dog']

for i, (name, label) in enumerate(zip(MODEL_ORDER, MODEL_LABELS)):
    cm_raw = models[name]['summary']['confusion_matrix_counts']
    cm = np.array(cm_raw)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    ax = axes[i]
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=100)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(classes, fontsize=9)
    ax.set_yticklabels(classes, fontsize=9)
    ax.set_xlabel('Predicted', fontsize=9)
    if i == 0:
        ax.set_ylabel('True', fontsize=9)
    acc = models[name]['summary']['final_test_accuracy'] * 100
    ax.set_title(f'{label.replace(chr(10), " ")}\n{acc:.2f}%', fontsize=9)

    for r in range(2):
        for c in range(2):
            color = 'white' if cm_norm[r, c] > 60 else 'black'
            ax.text(c, r, f'{cm[r, c]}\n({cm_norm[r, c]:.1f}%)',
                    ha='center', va='center', fontsize=8, color=color)

plt.suptitle('Section 6: Confusion Matrices — All Models (row-normalised, test set)', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUTS / 'deep_dive_confusion_matrices.png', bbox_inches='tight', dpi=120)
plt.show()

# Per-class accuracy
print(f'\n{'Model':<28} {'Cat Acc':>8} {'Dog Acc':>8} {'Errors':>8} {'Cat err':>8} {'Dog err':>8}')
print('-' * 70)
for name in MODEL_ORDER:
    cm = np.array(models[name]['summary']['confusion_matrix_counts'])
    cat_acc = cm[0,0] / cm[0,:].sum() * 100
    dog_acc = cm[1,1] / cm[1,:].sum() * 100
    cat_err = cm[0,1]
    dog_err = cm[1,0]
    total_err = cat_err + dog_err
    print(f'{name:<28} {cat_acc:>7.1f}% {dog_acc:>7.1f}% {total_err:>8} {cat_err:>8} {dog_err:>8}')

### Error Analysis

**Scratch CNN errors:**
- 545 cats predicted as dogs (46% error rate on cats!)
- 296 dogs predicted as cats (12% error rate on dogs)
- Strongly biased toward predicting "dog" — likely due to class imbalance (68% dogs in test set). Without class weights, the model learns it's safer to guess dog.

**Deeper CNN (with class weights):**
- Cat error drops dramatically: 86 / 1,183 = 7.3% (vs 46% without class weights!)
- Dog error: 134 / 2,486 = 5.4%
- Class weighting is transformative when classes are imbalanced.

**Transfer learning errors (ResNet50):**
- Only 3 cats misclassified as dogs (0.25% cat error)
- 11 dogs misclassified as cats (0.44% dog error)
- Total: 14 errors in 3,669 test images
- The remaining errors are likely genuinely ambiguous images (small puppies that look like kittens, or long-haired cats that look dog-like)

**MobileNetV3 asymmetry:**
- 58 cats → dog (4.9%) vs. 39 dogs → cat (1.6%)
- Smaller model still struggles slightly more with cats — the minority class always harder to learn without explicit weighting.

**Key lesson:** For imbalanced datasets, **always report macro F1** alongside accuracy. Accuracy can be misleading — a model that always predicts "dog" would get 67.8% accuracy on this test set while being completely useless for cat classification.

---
## 7. Full Visual Comparison

In [ ]:
# Display comparison report images if available
for name, label in zip(MODEL_ORDER, MODEL_LABELS):
    cm_img = COMP_DIR / name / 'confusion_matrix.png'
    if cm_img.exists():
        print(f'Confusion matrix available: {name}')

# Display loss curves from individual runs
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(20, 4))
for i, (name, label) in enumerate(zip(MODEL_ORDER, MODEL_LABELS)):
    hist = models[name]['training_history']
    eps = [e['epoch'] for e in hist]
    val_accs = [e['val_accuracy'] * 100 for e in hist]
    train_accs = [e['train_accuracy'] * 100 for e in hist]

    ax = axes[i]
    ax.plot(eps, train_accs, '--', color=MODEL_COLORS[i], linewidth=1.5, alpha=0.6, label='Train')
    ax.plot(eps, val_accs, '-', color=MODEL_COLORS[i], linewidth=2, label='Val')
    final_acc = models[name]['summary']['final_test_accuracy'] * 100
    ax.set_title(f'{label.replace(chr(10), " ")}\nTest: {final_acc:.2f}%', fontsize=9)
    ax.set_xlabel('Epoch', fontsize=8)
    ax.set_ylim(60, 105)
    if i == 0:
        ax.set_ylabel('Accuracy (%)', fontsize=8)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=8)

plt.suptitle('Section 7: Validation Accuracy Curves — All Models', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUTS / 'deep_dive_all_curves.png', bbox_inches='tight', dpi=120)
plt.show()

---
## 8. Conclusions

### Transfer Learning vs. Scratch — Summary

**The central question: why does a model pretrained on ImageNet (flowers, vehicles, animals) learn to classify cats vs. dogs better than one trained only on cat/dog data?**

> Transfer learning works because visual features are **hierarchical and universal**. The first layers of any CNN trained on natural images learn Gabor-like edge detectors — this is a mathematical property of natural image statistics, not a property of the specific training labels. Later layers build progressively abstract representations: textures, parts, objects. These representations are reusable across tasks because the *structure* of visual information is the same regardless of whether we're classifying dog breeds or car models.
>
> With only 3,312 training images, a scratch model can't learn reliable high-level representations — it sees too few examples of each visual pattern. A pretrained model has already seen millions of examples of those patterns in ImageNet, and the downstream task only needs to learn the final discrimination.

**Results summary:**

| Model | Test Acc | Macro F1 | Params | Train Time | Key technique |
|---|---|---|---|---|---|
| Scratch CNN | 77.08% | 72.1% | 422K | 21.4 min | Baseline |
| Deeper CNN | 94.00% | 93.2% | 3.5M | 41.2 min | AdamW + cosine + class weights |
| MobileNetV3 | 97.36% | 97.0% | 1.5M | 9.3 min | ImageNet transfer (efficient) |
| ResNet18 | 98.99% | 98.8% | 11.2M | 10.7 min | ImageNet transfer + skip connections |
| **ResNet50** | **99.62%** | **99.6%** | 23.5M | 13.6 min | ImageNet transfer + bottleneck blocks |

**Deployment recommendation:**
- **Best accuracy:** ResNet50 — 99.62%, 13.6 min training
- **Best efficiency:** MobileNetV3 — 97.36%, 9.3 min, 1.5M params (best for mobile/edge)
- **Best scratch model:** Deeper CNN — 94.00% with class weights and AdamW

**Key takeaways:**
1. On small datasets, **transfer learning is not an optimisation — it's a necessity**. The 22% gap between scratch (77%) and ResNet50 (99.6%) cannot be closed by model design alone.
2. **Class imbalance must be addressed explicitly.** Without class weights, the scratch model reaches 77% but ignores 46% of cat images — macro F1 reveals this when accuracy doesn't.
3. **Checkpoint selection by validation accuracy is critical.** Best performance often occurs before the final epoch; the final checkpoint can be 0.5–2% worse due to mild overfitting.
4. **More parameters ≠ better without pretrained weights.** The deeper scratch CNN at 3.5M params still loses to MobileNetV3 at 1.5M params with pretrained weights.